# Create final track-level train/validation/test assignments

This notebook creates the final `track_id,split` CSV for the 7,324-track experiment.

- Exact sizes: 5,127 train, 1,099 validation, 1,098 test
- Multi-label stratification over six target genres
- Seed: 42
- No artist or album grouping for the current experiments
- Every complete track, including all its log-Mel windows, belongs to one split


In [1]:
%pip install -q iterative-stratification


In [2]:
from pathlib import Path
import ast
import json
import re

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

SEED = 42
EXPECTED_TRACKS = 7_324
TARGET_GENRES = ["classical", "electronic", "folk", "hiphop", "jazz", "rock"]
EXPECTED_SIZES = {"train": 5_127, "validation": 1_099, "test": 1_098}


In [3]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Not running in Colab; using a local data directory if available.")

DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/music-genre-classification/data")
if DRIVE_DATA_ROOT.is_dir():
    DATA_ROOT = DRIVE_DATA_ROOT
else:
    DATA_ROOT = next(
        (p for p in (Path.cwd() / "data", Path.cwd().parent / "data")
         if (p / "split_csv.csv").is_file()),
        None,
    )

assert DATA_ROOT is not None, "Could not locate the project data directory."
METADATA_CSV = DATA_ROOT / "split_csv.csv"
GENRES_CSV = DATA_ROOT / "genres_df.csv"
OUTPUT_CSV = DATA_ROOT / "track_split_assignments.csv"

assert METADATA_CSV.is_file()
assert GENRES_CSV.is_file()
print("Data root:", DATA_ROOT)
print("Output:", OUTPUT_CSV)


Mounted at /content/drive
Data root: /content/drive/MyDrive/music-genre-classification/data
Output: /content/drive/MyDrive/music-genre-classification/data/track_split_assignments.csv


In [4]:
def canonical_track_id(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    text = re.sub(r"^track[_\-\s]*", "", text, flags=re.IGNORECASE)
    return str(int(text)) if re.fullmatch(r"\d+", text) else None


metadata = pd.read_csv(METADATA_CSV, dtype=str, keep_default_na=False)
genres = pd.read_csv(GENRES_CSV, dtype=str, keep_default_na=False)

assert "TRACK_ID" in metadata and "TRACK_ID" in genres
metadata["_key"] = metadata["TRACK_ID"].map(canonical_track_id)
genres["_key"] = genres["TRACK_ID"].map(canonical_track_id)

assert len(metadata) == EXPECTED_TRACKS
assert metadata["_key"].notna().all() and metadata["_key"].is_unique
assert genres["_key"].notna().all() and genres["_key"].is_unique
assert set(metadata["_key"]) == set(genres["_key"])


In [5]:
# Preserve metadata order and exact original TRACK_ID formatting.
working = metadata[["TRACK_ID", "_key"]].merge(
    genres[["_key"] + TARGET_GENRES], on="_key", how="left", validate="one_to_one"
)

Y = working[TARGET_GENRES].apply(pd.to_numeric, errors="raise").to_numpy(dtype=np.int8)
assert np.isin(Y, [0, 1]).all()
assert (Y.sum(axis=1) >= 1).all()

print("Tracks:", len(working))
display(pd.Series(Y.sum(axis=0), index=TARGET_GENRES, name="positives").to_frame())


Tracks: 7324


,positives
classical,1446
electronic,1733
folk,1323
hiphop,1304
jazz,1376
rock,1486


In [7]:
def enforce_exact_test_size(
    train_idx,
    test_idx,
    labels,
    required_test_size,
    seed,
):
    """
    Enforce the exact requested size while minimizing
    disturbance to the multi-label distribution.
    """
    train_idx = np.asarray(
        train_idx,
        dtype=int,
    )

    test_idx = np.asarray(
        test_idx,
        dtype=int,
    )

    labels = np.asarray(
        labels,
        dtype=float,
    )

    rng = np.random.default_rng(seed)

    target_positive_counts = (
        labels.sum(axis=0)
        * required_test_size
        / len(labels)
    )

    scales = np.maximum(
        target_positive_counts,
        1.0,
    )

    # Move tracks from train to test when test is too small.
    while len(test_idx) < required_test_size:
        current = labels[test_idx].sum(axis=0)

        candidates = train_idx.copy()
        rng.shuffle(candidates)

        proposed = (
            current[None, :]
            + labels[candidates]
        )

        scores = np.square(
            (
                proposed
                - target_positive_counts
            )
            / scales
        ).sum(axis=1)

        chosen = candidates[
            int(np.argmin(scores))
        ]

        train_idx = train_idx[
            train_idx != chosen
        ]

        test_idx = np.append(
            test_idx,
            chosen,
        )

    # Move tracks from test to train when test is too large.
    while len(test_idx) > required_test_size:
        current = labels[test_idx].sum(axis=0)

        candidates = test_idx.copy()
        rng.shuffle(candidates)

        proposed = (
            current[None, :]
            - labels[candidates]
        )

        scores = np.square(
            (
                proposed
                - target_positive_counts
            )
            / scales
        ).sum(axis=1)

        chosen = candidates[
            int(np.argmin(scores))
        ]

        test_idx = test_idx[
            test_idx != chosen
        ]

        train_idx = np.append(
            train_idx,
            chosen,
        )

    return (
        np.sort(train_idx),
        np.sort(test_idx),
    )

## Stage 1: 5,127 training tracks and a 2,197-track holdout


In [8]:
required_holdout_size = (
    EXPECTED_SIZES["validation"]
    + EXPECTED_SIZES["test"]
)

stage1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=required_holdout_size,
    random_state=SEED,
)

train_idx, holdout_idx = next(
    stage1.split(
        np.zeros(len(working)),
        Y,
    )
)

print(
    "Initial stage-1 sizes:",
    len(train_idx),
    len(holdout_idx),
)

train_idx, holdout_idx = (
    enforce_exact_test_size(
        train_idx=train_idx,
        test_idx=holdout_idx,
        labels=Y,
        required_test_size=required_holdout_size,
        seed=SEED,
    )
)

print(
    "Corrected stage-1 sizes:",
    len(train_idx),
    len(holdout_idx),
)

assert (
    len(train_idx)
    == EXPECTED_SIZES["train"]
)

assert (
    len(holdout_idx)
    == required_holdout_size
)

Initial stage-1 sizes: 5141 2183
Corrected stage-1 sizes: 5127 2197


## Stage 2: divide the holdout into validation and test


In [9]:
stage2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=EXPECTED_SIZES["test"],
    random_state=SEED + 1,
)

validation_local, test_local = next(
    stage2.split(
        np.zeros(len(holdout_idx)),
        Y[holdout_idx],
    )
)

print(
    "Initial stage-2 sizes:",
    len(validation_local),
    len(test_local),
)

validation_local, test_local = (
    enforce_exact_test_size(
        train_idx=validation_local,
        test_idx=test_local,
        labels=Y[holdout_idx],
        required_test_size=(
            EXPECTED_SIZES["test"]
        ),
        seed=SEED + 1,
    )
)

validation_idx = holdout_idx[
    validation_local
]

test_idx = holdout_idx[
    test_local
]

print(
    "Corrected stage-2 sizes:",
    len(validation_idx),
    len(test_idx),
)

assert (
    len(validation_idx)
    == EXPECTED_SIZES["validation"]
)

assert (
    len(test_idx)
    == EXPECTED_SIZES["test"]
)

Initial stage-2 sizes: 1102 1095
Corrected stage-2 sizes: 1099 1098


In [10]:
assignment = np.empty(len(working), dtype=object)
assignment[train_idx] = "train"
assignment[validation_idx] = "validation"
assignment[test_idx] = "test"

result = pd.DataFrame({
    "track_id": working["TRACK_ID"],
    "split": assignment,
})

display(result.head())
display(result["split"].value_counts().reindex(["train", "validation", "test"]).to_frame("tracks"))


,track_id,split
0,track_1147710,train
1,track_1103990,validation
2,track_1131488,train
3,track_1198188,train
4,track_0245962,test


,tracks
split,
train,5127
validation,1099
test,1098


## Validate separation and genre balance


In [12]:
# ==========================================================
# Validate assignments
# ==========================================================

assert len(result) == EXPECTED_TRACKS

assert result["track_id"].is_unique

assert result["split"].notna().all()

assert (
    result["split"]
    .value_counts()
    .to_dict()
    == EXPECTED_SIZES
)


split_sets = {
    name: set(
        result.loc[
            result["split"] == name,
            "track_id",
        ]
    )
    for name in EXPECTED_SIZES
}

assert split_sets["train"].isdisjoint(
    split_sets["validation"]
)

assert split_sets["train"].isdisjoint(
    split_sets["test"]
)

assert split_sets["validation"].isdisjoint(
    split_sets["test"]
)

assert (
    set().union(*split_sets.values())
    == set(metadata["TRACK_ID"])
)

print("PASS: assignments are unique and disjoint.")


# ==========================================================
# Construct a numeric audit table from validated matrix Y
# ==========================================================

audit = pd.DataFrame(
    Y,
    columns=TARGET_GENRES,
)

audit.insert(
    0,
    "TRACK_ID",
    working["TRACK_ID"].to_numpy(),
)

audit["split"] = assignment


# Confirm the genre columns really are numeric.
assert all(
    pd.api.types.is_numeric_dtype(
        audit[column]
    )
    for column in TARGET_GENRES
)


# ==========================================================
# Genre counts and prevalence
# ==========================================================

split_order = [
    "train",
    "validation",
    "test",
]

counts = (
    audit
    .groupby("split")[TARGET_GENRES]
    .sum()
    .reindex(split_order)
)

split_denominators = (
    audit["split"]
    .value_counts()
    .reindex(split_order)
)

prevalence = counts.div(
    split_denominators,
    axis=0,
)

overall = audit[
    TARGET_GENRES
].mean()

deviation = prevalence.subtract(
    overall,
    axis=1,
)


print("Positive-label counts:")
display(counts.astype(int))

print("Genre prevalence:")
display(prevalence.round(5))

print(
    "Absolute prevalence deviation "
    "from the full dataset:"
)
display(deviation.abs().round(5))

maximum_deviation = float(
    deviation.abs().to_numpy().max()
)

print(
    "Maximum absolute deviation:",
    maximum_deviation,
)


# A difference below 1 percentage point is an
# appropriately tight balance for this experiment.
assert maximum_deviation < 0.01, (
    "Genre balance differs by more than "
    "one percentage point."
)

print(
    "PASS: all split genre prevalences are "
    "within one percentage point of the "
    "complete dataset."
)

PASS: assignments are unique and disjoint.
Positive-label counts:


,classical,electronic,folk,hiphop,jazz,rock
split,,,,,,
train,1010,1209,924,911,961,1038
validation,218,262,199,196,207,224
test,218,262,200,197,208,224


Genre prevalence:


,classical,electronic,folk,hiphop,jazz,rock
split,,,,,,
train,0.19700,0.23581,0.18022,0.17769,0.18744,0.20246
validation,0.19836,0.23840,0.18107,0.17834,0.18835,0.20382
test,0.19854,0.23862,0.18215,0.17942,0.18944,0.20401


Absolute prevalence deviation from the full dataset:


,classical,electronic,folk,hiphop,jazz,rock
split,,,,,,
train,0.00044,0.00081,0.00042,0.00036,0.00044,0.00044
validation,0.00093,0.00178,0.00043,0.00030,0.00048,0.00093
test,0.00111,0.00200,0.00151,0.00137,0.00156,0.00111


Maximum absolute deviation: 0.0019963311477399492
PASS: all split genre prevalences are within one percentage point of the complete dataset.


## Determinism check


In [14]:
def recreate_assignment():
    # ======================================================
    # Recreate Stage 1
    # ======================================================

    required_holdout_size = (
        EXPECTED_SIZES["validation"]
        + EXPECTED_SIZES["test"]
    )

    stage1_recreated = (
        MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=required_holdout_size,
            random_state=SEED,
        )
    )

    train_recreated, holdout_recreated = next(
        stage1_recreated.split(
            np.zeros(len(working)),
            Y,
        )
    )

    train_recreated, holdout_recreated = (
        enforce_exact_test_size(
            train_idx=train_recreated,
            test_idx=holdout_recreated,
            labels=Y,
            required_test_size=(
                required_holdout_size
            ),
            seed=SEED,
        )
    )

    assert (
        len(train_recreated)
        == EXPECTED_SIZES["train"]
    )

    assert (
        len(holdout_recreated)
        == required_holdout_size
    )


    # ======================================================
    # Recreate Stage 2
    # ======================================================

    stage2_recreated = (
        MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=EXPECTED_SIZES["test"],
            random_state=SEED + 1,
        )
    )

    validation_local, test_local = next(
        stage2_recreated.split(
            np.zeros(
                len(holdout_recreated)
            ),
            Y[holdout_recreated],
        )
    )

    validation_local, test_local = (
        enforce_exact_test_size(
            train_idx=validation_local,
            test_idx=test_local,
            labels=Y[holdout_recreated],
            required_test_size=(
                EXPECTED_SIZES["test"]
            ),
            seed=SEED + 1,
        )
    )

    validation_recreated = (
        holdout_recreated[
            validation_local
        ]
    )

    test_recreated = (
        holdout_recreated[
            test_local
        ]
    )

    assert (
        len(validation_recreated)
        == EXPECTED_SIZES["validation"]
    )

    assert (
        len(test_recreated)
        == EXPECTED_SIZES["test"]
    )


    # ======================================================
    # Reconstruct assignment array
    # ======================================================

    recreated = np.empty(
        len(working),
        dtype=object,
    )

    recreated[train_recreated] = "train"

    recreated[
        validation_recreated
    ] = "validation"

    recreated[
        test_recreated
    ] = "test"

    return recreated

In [15]:
recreated_assignment = recreate_assignment()

differences = np.flatnonzero(
    assignment != recreated_assignment
)

print(
    "Different assignments:",
    len(differences),
)

assert np.array_equal(
    assignment,
    recreated_assignment,
), (
    f"Reproducibility check failed for "
    f"{len(differences)} tracks."
)

print(
    "PASS: seed, stratification, and exact-size "
    "correction reproduce identical assignments."
)

Different assignments: 0
PASS: seed, stratification, and exact-size correction reproduce identical assignments.


## Save the final two-column CSV


In [16]:
result.to_csv(OUTPUT_CSV, index=False)
saved = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)

assert list(saved.columns) == ["track_id", "split"]
assert len(saved) == EXPECTED_TRACKS
assert saved["track_id"].is_unique
assert saved["split"].value_counts().to_dict() == EXPECTED_SIZES

print("Saved:", OUTPUT_CSV)
print("Rows:", len(saved))
print(saved["split"].value_counts())


Saved: /content/drive/MyDrive/music-genre-classification/data/track_split_assignments.csv
Rows: 7324
split
train         5127
validation    1099
test          1098
Name: count, dtype: int64


In [17]:
# Optional Colab download.
try:
    from google.colab import files
    files.download(str(OUTPUT_CSV))
except ImportError:
    print("Output is available at:", OUTPUT_CSV)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>